In [1]:
import os
import pickle

from dotenv import load_dotenv 
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import SGDClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sqlalchemy import create_engine
from xgboost import XGBClassifier

In [2]:
load_dotenv()
engine = create_engine(os.getenv("DATABASE_URL"))
seed = 123

In [3]:

classifier_techniques = {

    "logistic_regression": {
        "method": SGDClassifier(
            loss="log_loss",
            penalty="elasticnet",
            random_state=seed
        ),
        "batch_training": True
    },

    "support_vector_machine": {
        "method": SGDClassifier(
            loss="hinge",
            penalty="elasticnet",
            random_state=seed
        ),
        "batch_training": True
    },

    "mlp_classifier": {
        "method": MLPClassifier(
            solver="adam",
            max_iter=500,
            random_state=seed
        ),
        "batch_training": True
    },

    "naive_bayes": {
        "method": GaussianNB(),
        "batch_training": True
    },

    "xgboost": {
        "method": XGBClassifier(
            objective="binary:logistic",
            eval_metric="auc",
            tree_method="hist",
            n_estimators=1000,
            random_state=seed,
            n_jobs=4
        ),
        "batch_training": False
    }

}


In [4]:
for name, technique in classifier_techniques.items():

    if name == "xgboost":

        technique["pipeline"] = Pipeline([
            ("classifier", technique["method"])
        ])

    else:

        technique["pipeline"] = Pipeline([
            ("scaler", StandardScaler()),
            ("classifier", technique["method"])
        ])

In [5]:
with open("../data/classifier_techniques.pkl", "wb") as file:
    pickle.dump(classifier_techniques, file)